# PromptTemplate
It allows you to create templates for prompts that can be filled in with specific variables at runtime. This is useful for generating dynamic prompts based on user input or other contextual information.

In [29]:
from langchain_core.prompts import PromptTemplate

template_1 = PromptTemplate(
    input_variables=["topic"],
    template="Explain the concept of {topic} in simple terms."
)
template_2 = PromptTemplate.from_template("Explain the concept of {topic} in simple terms."
)

formatted_prompt_1 = template_1.format(topic="machine learning")
formatted_prompt_2 = template_2.format(topic="machine learning")
print(formatted_prompt_1)
print(formatted_prompt_2)


template_1.invoke(input={"topic": "machine learning"})
template_2.invoke(input={"topic": "machine learning"})

Explain the concept of machine learning in simple terms.
Explain the concept of machine learning in simple terms.


StringPromptValue(text='Explain the concept of machine learning in simple terms.')

# ChatPromptTemplate
It allows you to define a structured prompt for chat-based interactions, where you can specify the roles (e.g., system, human, ai) and the content of each message. This is useful for creating prompts that guide the behavior of a language model in a conversational context.

In [30]:
from langchain_core.prompts import ChatPromptTemplate

template_1 = ChatPromptTemplate(messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain the concept of {topic} in simple terms."}
])

template_2 = ChatPromptTemplate.from_messages([
    ("system", "You are a professional polyglot AI. Translate the user's text into {language}. "
               "Return the output strictly in JSON format."),
    ("human", "{text}"),
])
# Using format method to create the final prompt
formatted_prompt_1 = template_1.format(topic="machine learning")
formatted_prompt_2 = template_2.format(language="Spanish", text="Hello, how are you?")
print(formatted_prompt_1)
print(formatted_prompt_2)

# Using invoke method to get the final prompt directly
template_1.invoke(input={"topic": "machine learning"})
template_2.invoke(input={"language": "Spanish", "text": "Hello, how are you?"})

System: You are a helpful assistant.
Human: Explain the concept of machine learning in simple terms.
System: You are a professional polyglot AI. Translate the user's text into Spanish. Return the output strictly in JSON format.
Human: Hello, how are you?


ChatPromptValue(messages=[SystemMessage(content="You are a professional polyglot AI. Translate the user's text into Spanish. Return the output strictly in JSON format.", additional_kwargs={}, response_metadata={}), HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})])

# MessagesPlaceholder
It allows you to include a placeholder in your prompt that can be filled with dynamic content at runtime, such as chat history or other relevant information.

In [31]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful technical architect. Answer briefly."),
    MessagesPlaceholder(variable_name="chat_history"), 
    ("human", "{input}"),
])

demo_ephemeral_history = [
    HumanMessage(content="Hi, I'm building a microservices architecture."),
    AIMessage(content="Nice! Are you using REST or gRPC for inter-service communication?")
]

prompt.invoke(input={"chat_history": demo_ephemeral_history,"input": "What are the best practices for designing microservices?"})

ChatPromptValue(messages=[SystemMessage(content='You are a helpful technical architect. Answer briefly.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Hi, I'm building a microservices architecture.", additional_kwargs={}, response_metadata={}), AIMessage(content='Nice! Are you using REST or gRPC for inter-service communication?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What are the best practices for designing microservices?', additional_kwargs={}, response_metadata={})])

# FewShotChatMessagePromptTemplate
It allows you to create a few-shot learning prompt for chat-based interactions, where you can provide examples of input-output pairs to guide the model's responses. This is useful for improving the model's performance on specific tasks by showing it how to respond to similar inputs.

In [33]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder

examples = [
    {"input": "2+2", "output": "The math checks out: it's 4."},
    {"input": "Who is the CEO?", "output": "I'm a math bot, not a LinkedIn scraper!"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a sassy math assistant."),
    few_shot_prompt,
    MessagesPlaceholder(variable_name="history"), # For chat memory
    ("human", "{input}"),
])

final_prompt.invoke(input={"history": demo_ephemeral_history, "input": "What is 3*3?"})

ChatPromptValue(messages=[SystemMessage(content='You are a sassy math assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='2+2', additional_kwargs={}, response_metadata={}), AIMessage(content="The math checks out: it's 4.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Who is the CEO?', additional_kwargs={}, response_metadata={}), AIMessage(content="I'm a math bot, not a LinkedIn scraper!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Hi, I'm building a microservices architecture.", additional_kwargs={}, response_metadata={}), AIMessage(content='Nice! Are you using REST or gRPC for inter-service communication?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is 3*3?', additional_kwargs={}, response_metadata={})])

# FewShotPromptTemplate
It allows you to create a few-shot learning prompt for non-chat interactions, where you can provide examples of input-output pairs to guide the model's responses. This is useful for improving the model's performance on specific tasks by showing it how to respond to similar inputs in a non-conversational context.

In [34]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

# 1. Define your examples (The "Shots")
examples = [
    {"input": "The server crashed again.", "output": "CATEGORY: INFRA | SEVERITY: HIGH"},
    {"input": "The UI button is slightly off-center.", "output": "CATEGORY: UX | SEVERITY: LOW"},
    {"input": "Users can't login with SSO.", "output": "CATEGORY: AUTH | SEVERITY: CRITICAL"},
]

# 2. Define how each example should be formatted
# This template acts as the "mini-mold" for each item in the list above
example_formatter = PromptTemplate(
    input_variables=["input", "output"],
    template="Input: {input}\nOutput: {output}"
)

# 3. Create the FewShotPromptTemplate
few_shot_prompt = FewShotPromptTemplate(
    examples=examples, 
    example_prompt=example_formatter, 
    prefix="Classify the following technical issues based on department and urgency:",
    suffix="Input: {user_query}\nOutput:", 
    input_variables=["user_query"],
    example_separator="\n\n" # Clearly separates the shots
)

# 4. Usage
final_prompt = few_shot_prompt.format(user_query="The database is slow during peak hours.")
print(final_prompt)

Classify the following technical issues based on department and urgency:

Input: The server crashed again.
Output: CATEGORY: INFRA | SEVERITY: HIGH

Input: The UI button is slightly off-center.
Output: CATEGORY: UX | SEVERITY: LOW

Input: Users can't login with SSO.
Output: CATEGORY: AUTH | SEVERITY: CRITICAL

Input: The database is slow during peak hours.
Output:
